# Session 8: Regression, Interactions & Conditional Prediction

**Bayesian Analysis of Empirical Data (2026)**  
*Author: Irina Knyazeva*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iknyazeva/bayes-cogsci-book/blob/main/notebooks/colab/08_regression_interactions_prediction.ipynb)

### Obligatory Graded Assessment: Task 2 Vertical Slice
This notebook walks through the complete vertical slice for Bayesian linear regression with centering, interaction modeling, and natural outcome-scale prediction using the canonical **KidIQ** dataset (Gelman et al., *ROS* Ch. 10–11).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arviz as az
import pymc as pm

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
rng = np.random.default_rng(2026)

# Synthetic replica of documented KidIQ dataset (N = 434)
N = 434
mom_hs = rng.binomial(1, 0.78, size=N)
mom_iq = rng.normal(100, 15, size=N)
# Generative parameters: alpha=82, beta_iq=0.56, beta_hs=5.5, beta_inter=-0.48, sigma=18
mu_true = 82 + 0.56 * (mom_iq - 100) + 5.5 * mom_hs - 0.48 * (mom_iq - 100) * mom_hs
kid_score = rng.normal(mu_true, 18)

df = pd.DataFrame({"kid_score": kid_score, "mom_iq": mom_iq, "mom_hs": mom_hs})
df["mom_iq_c"] = df["mom_iq"] - 100.0  # Substantive centering at population IQ 100
df.head()

,kid_score,mom_iq,mom_hs,mom_iq_c
0,70.181200,107.371243,1,7.371243
1,121.830958,110.219754,1,10.219754
2,119.190362,107.994288,1,7.994288
3,79.073183,111.265698,1,11.265698
4,66.877069,89.664408,1,-10.335592


## 1. Specifying and Fitting the Centered Interaction Model in PyMC

In [2]:
with pm.Model() as kidiq_model:
    # Scale-aware weakly informative priors
    alpha = pm.Normal("alpha", mu=80.0, sigma=15.0)
    beta_iq = pm.Normal("beta_iq", mu=0.5, sigma=0.5)
    beta_hs = pm.Normal("beta_hs", mu=5.0, sigma=10.0)
    beta_inter = pm.Normal("beta_inter", mu=0.0, sigma=0.5)
    sigma = pm.HalfNormal("sigma", sigma=20.0)
    
    # Linear predictor with interaction
    mu = alpha + beta_iq * df["mom_iq_c"].values + beta_hs * df["mom_hs"].values + beta_inter * (df["mom_iq_c"].values * df["mom_hs"].values)
    y_obs = pm.Normal("kid_score", mu=mu, sigma=sigma, observed=df["kid_score"].values)
    
    # Sample using NUTS
    idata_kidiq = pm.sample(draws=1000, tune=1000, chains=4, random_seed=2026, return_inferencedata=True)

# Diagnostic check
print(az.summary(idata_kidiq, var_names=["alpha", "beta_iq", "beta_hs", "beta_inter", "sigma"])[["mean", "sd", "hdi_3%", "hdi_97%", "r_hat"]])

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [alpha, beta_iq, beta_hs, beta_inter, sigma]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 1 seconds.


              mean     sd  hdi_3%  hdi_97%  r_hat
alpha       80.960  1.819  77.611   84.313    1.0
beta_iq      0.513  0.117   0.290    0.732    1.0
beta_hs      5.902  2.037   1.977    9.518    1.0
beta_inter  -0.377  0.131  -0.618   -0.118    1.0
sigma       17.619  0.593  16.515   18.734    1.0


## 2. Natural Outcome-Scale Contrasts & Conditional Slopes

In [3]:
post = idata_kidiq.posterior

# Conditional Slopes:
slope_no_hs = post["beta_iq"].values.flatten()
slope_hs = (post["beta_iq"] + post["beta_inter"]).values.flatten()

# Outcome-scale high-school contrast at IQ = 80, 100, 120:
diff_at_80 = (post["beta_hs"] + post["beta_inter"] * (-20)).values.flatten()
diff_at_100 = post["beta_hs"].values.flatten()
diff_at_120 = (post["beta_hs"] + post["beta_inter"] * (20)).values.flatten()

print(f"Maternal IQ Slope (No HS): {slope_no_hs.mean():.2f} [{np.percentile(slope_no_hs, 2.5):.2f}, {np.percentile(slope_no_hs, 97.5):.2f}]")
print(f"Maternal IQ Slope (HS):    {slope_hs.mean():.2f} [{np.percentile(slope_hs, 2.5):.2f}, {np.percentile(slope_hs, 97.5):.2f}]")
print(f"\nHS Score Difference at IQ=80:  {diff_at_80.mean():.2f} points")
print(f"HS Score Difference at IQ=100: {diff_at_100.mean():.2f} points")
print(f"HS Score Difference at IQ=120: {diff_at_120.mean():.2f} points")

Maternal IQ Slope (No HS): 0.51 [0.29, 0.75]
Maternal IQ Slope (HS):    0.14 [0.01, 0.26]

HS Score Difference at IQ=80:  13.44 points
HS Score Difference at IQ=100: 5.90 points
HS Score Difference at IQ=120: -1.64 points
